## <span style="color: Gold"> **Multi Modal**

- 작동원리
    - 이미지를 패치로 나누기  ex) 224x224 ----> 16x16 ===> 196개 패치
    - 패치를 벡터로 변환
    - 위치 정보를 추가 : 각 패치가 이미지의 어디에 위치했는지 정보를 추가
    - transformer 처리 - 학습
    - 분류

- CNN과 차이점 : CNN은 특정 이미지의 부분을 보는 관점, multi modal은 vit 이미지 전체를 한번에 볼 수 있음
- 가상환경은 **python 3.10** 이 가장 안정적임

In [ ]:
# 라이브러리 설치 확인
import torch
print(torch.__version__)

2.9.1+cpu


In [ ]:
!conda list

In [ ]:
import torch
import requests
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForImageClassification

# 모델 이미지 프로세스 로드
model_name = 'google/vit-base-patch16-224'
image_processor = AutoImageProcessor.from_pretrained(
    model_name,
    use_fast = True
)

model = AutoModelForImageClassification.from_pretrained(
    model_name,
    device_map = 'auto'  #자동으로 CPU, GPU 선택
)


# 이미지 로드
image_urls = [
        "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg",
        "http://images.cocodataset.org/val2017/000000039769.jpg"
]

for idx, url in enumerate(image_urls, 1):
    try:
        # 이미지 다운로드
        image = Image.open(requests.get(url, stream=True).raw)
        print(f'이미지 크기: {image.size}')
        # 이미지 전처리
        inputs = image_processor(image, return_tensors='pt').to(model.device)
        print(f"전처리 후 텐서의 크기 : {inputs['pixel_values'].shape}")
                    # 출력 결과 ==> 전처리 후 텐서의 크기 : torch.Size([1, 3, 224, 224]) --> 1:배치크기, 3:
        # 추론
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
        # 결과를 해석
        predicted_class_id = logits.argmax(dim=-1).item()
        predicted_class_label = model.config.id2label[predicted_class_id]  #id2label : 토크나이저되면 숫자나오듯이 되는???
        confidence = torch.softmax(logits, dim=-1)[0][predicted_class_id].item()
        print(f'========= 예측결과 {idx} =============')
        print(f'클래스 ID: {predicted_class_id}')
        print(f'클래스 label: {predicted_class_label}')
        print(f'확률(신뢰도): {confidence}')
        # top5 예측결과
        probs = torch.softmax(logits, dim=-1)[0]
        top5_probs, top5_indices = torch.topk(probs, 5)
        for i, (prob, idx) in enumerate(zip(top5_probs,top5_indices),1):
            label = model.config.id2label[idx.item()]
            print(f' {i}에 해당하는 label"{label}"은 : {prob.item():.2f}')
                    # 출력 결과 ==>  1에 해당하는 label"lynx, catamount"은 : 0.44 ==> 상기 이미지가 이걸로 분류함
    except Exception as e:
        print(f'오류발생 {e}')

이미지 크기: (960, 686)
전처리 후 텐서의 크기 : torch.Size([1, 3, 224, 224])
========= 예측결과 1 =============
클래스 ID: 287
클래스 label: lynx, catamount
확률(신뢰도): 0.4407151937484741
 1에 해당하는 label"lynx, catamount"은 : 0.44
 2에 해당하는 label"cougar, puma, catamount, mountain lion, painter, panther, Felis concolor"은 : 0.03
 3에 해당하는 label"snow leopard, ounce, Panthera uncia"은 : 0.03
 4에 해당하는 label"Egyptian cat"은 : 0.02
 5에 해당하는 label"tiger cat"은 : 0.02
이미지 크기: (640, 480)
전처리 후 텐서의 크기 : torch.Size([1, 3, 224, 224])
========= 예측결과 2 =============
클래스 ID: 285
클래스 label: Egyptian cat
확률(신뢰도): 0.9374898076057434
 1에 해당하는 label"Egyptian cat"은 : 0.94
 2에 해당하는 label"tabby, tabby cat"은 : 0.04
 3에 해당하는 label"tiger cat"은 : 0.01
 4에 해당하는 label"lynx, catamount"은 : 0.00
 5에 해당하는 label"Siamese cat, Siamese"은 : 0.00
오류발생 cannot identify image file <_io.BytesIO object at 0x000001D439625490>


In [19]:
# vit pipline
import torch
from transformers import pipeline
classifier = pipeline(
    task = 'image-classification',
    model = 'google/vit-base-patch16-224',
    device = 0 if torch.cuda.is_available() else -1
)

test_images = [
        {
            "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg",
            "description": "고양이 이미지"
        },
        {
            "url": "http://images.cocodataset.org/val2017/000000039769.jpg",
            "description": "고양이 2마리 이미지"
        },
        {
            "url": "https://huggingface.co/datasets/Narsil/image_dummy/raw/main/parrots.png",
            "description": "앵무새 이미지"
        }
    ]

for idx, img_info in enumerate(test_images, 1):
    print(f"--{idx} : {img_info['description']}--")
    try:
        results = classifier(img_info['url'])  #label score
        print(f'예측 결과 : {len(results)}')
        for idx, result in enumerate(results, 1):
            print(f" {idx} label : {result['label']} / score : {result['score']:.2f}")
    except Exeption as e:
        print(f'error : {e}')

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.
Device set to use cpu


--1 : 고양이 이미지--
예측 결과 : 5
 1 label : lynx, catamount / score : 0.44
 2 label : cougar, puma, catamount, mountain lion, painter, panther, Felis concolor / score : 0.03
 3 label : snow leopard, ounce, Panthera uncia / score : 0.03
 4 label : Egyptian cat / score : 0.02
 5 label : tiger cat / score : 0.02
--2 : 고양이 2마리 이미지--
예측 결과 : 5
 1 label : Egyptian cat / score : 0.94
 2 label : tabby, tabby cat / score : 0.04
 3 label : tiger cat / score : 0.01
 4 label : lynx, catamount / score : 0.00
 5 label : Siamese cat, Siamese / score : 0.00
--3 : 앵무새 이미지--
예측 결과 : 5
 1 label : macaw / score : 0.99
 2 label : African grey, African gray, Psittacus erithacus / score : 0.01
 3 label : toucan / score : 0.00
 4 label : sulphur-crested cockatoo, Kakatoe galerita, Cacatua galerita / score : 0.00
 5 label : lorikeet / score : 0.00
